In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
ROOT_DRIVE = "/content/drive/MyDrive/gen-poem"
TRAIN_FILE = f"{ROOT_DRIVE}/data/train.txt"
TEST_FILE  = f"{ROOT_DRIVE}/data/test.txt"
CHECKPOINT_DIR = f"{ROOT_DRIVE}/checkpoints"
LORA_MODEL = f"{ROOT_DRIVE}/models/llama32-lucbat-lora"
MERGED_MODEL = f"{ROOT_DRIVE}/models/llama32-lucbat"

In [3]:
!pip install -q transformers datasets accelerate bitsandbytes peft sentencepiece safetensors
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 25.7 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
from huggingface_hub import login, whoami
from google.colab import userdata

try:
    login(token=userdata.get("HF_TOKEN"))
    print(whoami())
except Exception as e:
    print("Vui lòng thiết lập HF_TOKEN trong Colab Secrets.")

{'type': 'user', 'id': '6a53fd1cbc579eda5e1846ef', 'name': 'ntntloan', 'fullname': 'Nguyễn Trương Ngọc Thảo Loan', 'email': 'ntntloan@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/5f8f1bc8ca982ea73cc0ccbce2c6658b.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'my-token', 'role': 'read', 'createdAt': '2026-07-17T01:07:28.867Z'}}}


Load tokenizer, model

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name="meta-llama/Llama-3.2-3B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Cấu hình nén 4-bit
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load model 4bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    attn_implementation="sdpa"
)

# Bắt buộc khi dùng Gradient Checkpointing trong TrainingArguments
model.config.use_cache = False
model.gradient_checkpointing_enable()

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Prepare dataset

In [6]:
import os
import json

TRAIN_JSON = TRAIN_FILE.replace(".txt", ".json")
TEST_JSON = TEST_FILE.replace(".txt", ".json")

def process_file(txt_path, json_path, label):
    with open(txt_path, "r", encoding="utf-8") as f:
        # Đọc toàn bộ file và tách thành danh sách các bài thơ dựa trên 2 dấu xuống dòng
        raw_content = f.read().strip()
        poems = [p.strip() for p in raw_content.split("\n\n") if p.strip()]

    # Gom tất cả các bài thơ thành một danh sách các dictionary
    data_list = [{"text": poem} for poem in poems]

    # Ghi toàn bộ danh sách vào file JSON
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

    print(f"Đã chuyển đổi tập {label.upper()} thành công!")
    print(f"File JSON mới: {json_path} (Tổng số bài: {len(data_list)})")

process_file(TRAIN_FILE, TRAIN_JSON, "train")
process_file(TEST_FILE, TEST_JSON, "test")

Đã chuyển đổi tập TRAIN thành công!
File JSON mới: /content/drive/MyDrive/gen-poem/data/train.json (Tổng số bài: 67754)
Đã chuyển đổi tập TEST thành công!
File JSON mới: /content/drive/MyDrive/gen-poem/data/test.json (Tổng số bài: 38014)


Load dataset

In [7]:
from datasets import load_dataset

# Tập train, test
dataset=load_dataset("json", data_files={"train": TRAIN_JSON, "test": TEST_JSON,},)

# Tập eval
num_eval = min(1000, len(dataset["test"]))
eval = dataset["test"].select(range(num_eval))

# Tập debug: small_train, small_eval
#num_train_samples = min(100, len(dataset["train"]))
#num_eval_samples = min(20, len(dataset["test"]))
#small_train = dataset["train"].select(range(num_train_samples))
#small_eval = dataset["test"].select(range(num_eval_samples))

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Tokenize dữ liệu có Padding & Truncation

In [8]:
# Cấu hình chiều dài tối đa cho câu
max_length = 1024

def tokenize_function(examples):
    # Thêm EOS token vào cuối mỗi văn bản
    texts = [text + tokenizer.eos_token for text in examples["text"]]

    # Tokenize thông thường có truncation
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding=False # Sẽ để Data Collator tự động pad theo batch để tiết kiệm bộ nhớ
    )

    return tokenized

# Áp dụng Tokenize thông thường
tokenized_train = dataset["train"].map(tokenize_function, batched=True, remove_columns=["text"], desc="Tokenizing train dataset")
tokenized_eval = eval.map(tokenize_function, batched=True, remove_columns=["text"], desc="Tokenizing eval dataset")

# Các tập nhỏ để debug (nếu cần)
#tokenized_small_train = small_train.map(tokenize_function, batched=True, remove_columns=["text"], desc="Tokenizing small train dataset")
#tokenized_small_eval = small_eval.map(tokenize_function, batched=True, remove_columns=["text"], desc="Tokenizing small eval dataset")


Tokenizing train dataset:   0%|          | 0/67754 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

LoRA

In [9]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Chuẩn bị model cho k-bit training
model = prepare_model_for_kbit_training(model)

# Tạo LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", # Các tầng xử lý ngữ cảnh (Attention)
        "gate_proj", "up_proj", "down_proj"     # Các tầng xử lý tri thức (Feed-Forward)
    ]
)

# Gắn LoRA
model = get_peft_model(model, lora_config)

# Hiển thị số lượng tham số được huấn luyện (LoRA) và tổng số tham số của mô hình
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Training Arguments

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir=CHECKPOINT_DIR,  # Thư mục lưu checkpoint (nên trỏ thẳng lên Google Drive)
    dataloader_num_workers=2,   # Sử dụng 2 luồng CPU nạp dữ liệu song song, giữ cho GPU không bị "đói" dữ liệu

    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit", # Bộ tối ưu hóa lượng hóa 8-bit, tiết kiệm 75% VRAM giúp chống tràn bộ nhớ trên T4
    report_to="none",         # Tắt đồng bộ bên thứ ba (WandB, Tensorboard) để giảm độ trễ mạng và nhẹ code

    num_train_epochs=2,            # Số lần mô hình học lặp lại toàn bộ tập dữ liệu (2 vòng là vừa đủ)
    weight_decay=0.01,             # Tránh hiện tượng overfitting (học vẹt dữ liệu)
    per_device_train_batch_size=4, # Số lượng mẫu từ tập Train được đẩy vào GPU xử lý cùng một lúc
    per_device_eval_batch_size=4,  # Số lượng mẫu từ tập Test/Valid được xử lý cùng một lúc khi làm bài kiểm tra
    gradient_accumulation_steps=4, # Cập nhật trọng số ngay sau mỗi batch để đạt tốc độ lặp (it/s) nhanh nhất
    gradient_checkpointing=True,

    learning_rate=2e-4,         # Tốc độ học tối ưu khi áp dụng LoRA cho các mô hình ngôn ngữ nhỏ 1B-3B
    lr_scheduler_type="cosine", # Giảm tốc độ học dần về cuối theo đường cong Cosine để dò điểm tối ưu chính xác
    warmup_steps=100,           # Tăng dần tốc độ học trong 100 bước đầu tiên để mô hình không bị "sốc" dữ liệu mới
    logging_steps=50,           # Cứ sau 50 bước huấn luyện sẽ in chỉ số Loss (mức độ sai số) ra màn hình để theo dõi

    eval_strategy="steps", # Bật tính năng tự động đánh giá định kỳ dựa trên số bước
    eval_steps=100,        # Cứ sau 100 steps thì chạy đánh giá tập test một lần

    save_strategy="steps", # Bật tính năng tự động lưu checkpoint định kỳ dựa trên số bước (steps)
    save_steps=100,        # Cứ sau 100 bước lặp thì lưu checkpoint một lần (Bắt buộc trùng với eval_steps)
    save_total_limit=2,    # Giới hạn số lượng checkpoint tối đa được lưu trên ổ cứng (GG Drive) tại một thời điểm

    metric_for_best_model="eval_loss",  # Chọn điểm lỗi trên tập test làm tiêu chí đánh giá mô hình hay/dở
    greater_is_better=False,            # Điểm lỗi (Loss) càng thấp (nhỏ) chứng tỏ mô hình làm thơ càng chuẩn luật
    load_best_model_at_end=True,        # Tự động tải lại checkpoint có loss thấp nhất đè lên mô hình khi kết thúc train
)

Trainer

In [11]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,

    processing_class=tokenizer,
    data_collator=data_collator,
)

Train

In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f"Resume từ {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Train từ đầu.")
    trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Train từ đầu.


Step,Training Loss,Validation Loss
100,3.810536,3.956053
200,3.438449,3.657619
300,3.271929,3.541397
400,3.209054,3.461373
500,3.131576,3.422158
600,3.093024,3.380774
700,3.033510,3.356641


In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f"Resume từ {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Train từ đầu.")
    trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Resume từ /content/drive/MyDrive/gen-poem/checkpoints/checkpoint-700


Step,Training Loss,Validation Loss
800,3.017712,3.344611
900,3.000744,3.308261
1000,2.951595,3.291997


Step,Training Loss,Validation Loss
800,3.017712,3.344611
900,3.000744,3.308261
1000,2.951595,3.291997
1100,2.935426,3.273497
1200,2.929963,3.253046
1300,2.880485,3.240682
1400,2.894065,3.220009
1500,2.843077,3.213905
1600,2.851446,3.189154
1700,2.841367,3.193846


In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f"Resume từ {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Train từ đầu.")
    trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Resume từ /content/drive/MyDrive/gen-poem/checkpoints/checkpoint-3300


Step,Training Loss,Validation Loss
3400,2.686278,3.072005
3500,2.630568,3.069623
3600,2.656738,3.065059
3700,2.645978,3.062015
3800,2.601829,3.061267
3900,2.627480,3.049924
4000,2.629104,3.046917
4100,2.601112,3.049065
4200,2.598741,3.043989
4300,2.441048,3.057662


In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f"Resume từ {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Train từ đầu.")
    trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Resume từ /content/drive/MyDrive/gen-poem/checkpoints/checkpoint-5300


Step,Training Loss,Validation Loss
5400,2.399223,3.036791
5500,2.393736,3.030615
5600,2.384350,3.028663
5700,2.393593,3.029195
5800,2.389546,3.025692
5900,2.405915,3.027342
6000,2.409675,3.022330
6100,2.375749,3.015272
6200,2.397028,3.012769
6300,2.364277,3.015325


In [12]:
import os
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if last_checkpoint is not None:
    print(f"Resume từ {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Train từ đầu.")
    trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Resume từ /content/drive/MyDrive/gen-poem/checkpoints/checkpoint-7200


Step,Training Loss,Validation Loss
7300,2.343217,3.005309
7400,2.355255,3.004676
7500,2.364904,3.001896
7600,2.335414,3.001411
7700,2.338439,3.001836
7800,2.338103,3.001209
7900,2.338002,3.002643
8000,2.355885,3.001915
8100,2.362646,3.002280
8200,2.354784,3.001664


In [15]:
print(trainer.state.best_model_checkpoint)

/content/drive/MyDrive/gen-poem/checkpoints/checkpoint-7800


Lưu LoRA Adapter

In [13]:
trainer.model.save_pretrained(LORA_MODEL)
tokenizer.save_pretrained(LORA_MODEL)

('/content/drive/MyDrive/gen-poem/models/llama32-lucbat-lora/tokenizer_config.json',
 '/content/drive/MyDrive/gen-poem/models/llama32-lucbat-lora/tokenizer.json')

Merge LoRA Adapter + Base Model

In [14]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL)

# Load LoRA Adapter
model = AutoPeftModelForCausalLM.from_pretrained(
    LORA_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

# Merge LoRA Adapter vào Base model
merged_model = model.merge_and_unload()

# Lưu mô hình hoàn chỉnh
merged_model.save_pretrained(MERGED_MODEL)
tokenizer.save_pretrained(MERGED_MODEL)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/gen-poem/models/llama32-lucbat/tokenizer_config.json',
 '/content/drive/MyDrive/gen-poem/models/llama32-lucbat/tokenizer.json')

Zip Merged Model

In [ ]:
ZIP_FILE = f"{ROOT_DRIVE}/models/llama32-lucbat.zip"
!zip -r "{ZIP_FILE}" "{MERGED_MODEL}"

Donwload Merged Model

In [ ]:
from google.colab import files
files.download(ZIP_FILE)